# AtlasVision AI
## Genetic Algorithm-Assisted CNN for Mammography Classification

**Objectif :** construire un prototype de classification assistée des mammographies par Deep Learning, avec optimisation automatique des hyperparamètres par algorithme génétique et visualisation explicable par Grad-CAM.

> Ce notebook est conçu pour un dépôt GitHub de candidature à un hackathon en IA médicale et Computer Vision.

### Classes possibles
- `normal`
- `benign`
- `malignant`

### Structure attendue des données
```text
data/
├── train/
│   ├── normal/
│   ├── benign/
│   └── malignant/
├── val/
│   ├── normal/
│   ├── benign/
│   └── malignant/
└── test/
    ├── normal/
    ├── benign/
    └── malignant/
```

## 1. Installation des dépendances

À exécuter une seule fois si les bibliothèques ne sont pas encore installées.

In [ ]:
# !pip install torch torchvision torchaudio scikit-learn matplotlib pandas numpy tqdm opencv-python pillow

## 2. Importation des bibliothèques

In [ ]:
import os
import random
import copy
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
from tqdm import tqdm

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
from torchvision import datasets, transforms, models

from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay, roc_auc_score

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

## 3. Paramètres généraux

In [ ]:
DATA_DIR = Path("data")
TRAIN_DIR = DATA_DIR / "train"
VAL_DIR = DATA_DIR / "val"
TEST_DIR = DATA_DIR / "test"

IMG_SIZE = 224
NUM_CLASSES = 3
CLASS_NAMES = ["normal", "benign", "malignant"]

RESULTS_DIR = Path("results")
RESULTS_DIR.mkdir(exist_ok=True)

## 4. Prétraitement et augmentation des images

Pour un prototype GitHub, les transformations suivantes sont suffisantes. Dans un vrai projet clinique, il faut ajouter une étape plus spécifique : normalisation du contraste, suppression des artefacts, contrôle qualité et validation radiologique.

In [ ]:
train_transforms = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomRotation(10),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225])
])

val_test_transforms = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225])
])

## 5. Chargement des données

Si vous n'avez pas encore placé les données dans le dossier `data/`, cette cellule affichera un message d'alerte. Le reste du notebook sert de squelette fonctionnel.

In [ ]:
def load_datasets(batch_size=16):
    if not TRAIN_DIR.exists() or not VAL_DIR.exists() or not TEST_DIR.exists():
        raise FileNotFoundError(
            "Veuillez créer les dossiers data/train, data/val et data/test avec les sous-dossiers de classes."
        )

    train_dataset = datasets.ImageFolder(TRAIN_DIR, transform=train_transforms)
    val_dataset = datasets.ImageFolder(VAL_DIR, transform=val_test_transforms)
    test_dataset = datasets.ImageFolder(TEST_DIR, transform=val_test_transforms)

    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, num_workers=2)
    val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False, num_workers=2)
    test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False, num_workers=2)

    return train_dataset, val_dataset, test_dataset, train_loader, val_loader, test_loader

## 6. Modèle CNN de base avec transfert d'apprentissage

Nous utilisons `ResNet18` pré-entraîné comme base robuste. L'algorithme génétique optimisera certains hyperparamètres : learning rate, dropout, batch size et stratégie de gel des couches.

In [ ]:
def build_model(dropout=0.3, freeze_backbone=True):
    model = models.resnet18(weights=models.ResNet18_Weights.IMAGENET1K_V1)

    if freeze_backbone:
        for param in model.parameters():
            param.requires_grad = False

    in_features = model.fc.in_features
    model.fc = nn.Sequential(
        nn.Dropout(dropout),
        nn.Linear(in_features, NUM_CLASSES)
    )
    return model.to(device)

## 7. Fonctions d'entraînement et de validation

In [ ]:
def train_one_epoch(model, loader, criterion, optimizer):
    model.train()
    running_loss, correct, total = 0.0, 0, 0

    for images, labels in loader:
        images, labels = images.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item() * images.size(0)
        preds = outputs.argmax(dim=1)
        correct += (preds == labels).sum().item()
        total += labels.size(0)

    return running_loss / total, correct / total


def evaluate(model, loader, criterion):
    model.eval()
    running_loss, correct, total = 0.0, 0, 0
    all_preds, all_labels = [], []

    with torch.no_grad():
        for images, labels in loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            loss = criterion(outputs, labels)
            running_loss += loss.item() * images.size(0)
            preds = outputs.argmax(dim=1)
            correct += (preds == labels).sum().item()
            total += labels.size(0)
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())

    return running_loss / total, correct / total, np.array(all_labels), np.array(all_preds)

## 8. Algorithme génétique pour optimiser le CNN

Chaque individu représente une combinaison d'hyperparamètres :

- `learning_rate`
- `dropout`
- `batch_size`
- `freeze_backbone`

La fitness est l'accuracy de validation après un entraînement court.

In [ ]:
SEARCH_SPACE = {
    "learning_rate": [1e-4, 3e-4, 1e-3],
    "dropout": [0.2, 0.3, 0.5],
    "batch_size": [8, 16, 32],
    "freeze_backbone": [True, False]
}

def random_individual():
    return {k: random.choice(v) for k, v in SEARCH_SPACE.items()}


def mutate(individual, mutation_rate=0.25):
    child = individual.copy()
    for key in child:
        if random.random() < mutation_rate:
            child[key] = random.choice(SEARCH_SPACE[key])
    return child


def crossover(parent1, parent2):
    child = {}
    for key in parent1:
        child[key] = parent1[key] if random.random() < 0.5 else parent2[key]
    return child


def fitness(individual, epochs=2):
    _, _, _, train_loader, val_loader, _ = load_datasets(batch_size=individual["batch_size"])
    model = build_model(dropout=individual["dropout"], freeze_backbone=individual["freeze_backbone"])
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(filter(lambda p: p.requires_grad, model.parameters()), lr=individual["learning_rate"])

    for epoch in range(epochs):
        train_one_epoch(model, train_loader, criterion, optimizer)

    _, val_acc, _, _ = evaluate(model, val_loader, criterion)
    return val_acc


def genetic_search(population_size=6, generations=3, elite_size=2):
    population = [random_individual() for _ in range(population_size)]
    history = []

    for gen in range(generations):
        print(f"Generation {gen+1}/{generations}")
        scored = []
        for individual in population:
            score = fitness(individual, epochs=2)
            scored.append((score, individual))
            print(score, individual)

        scored = sorted(scored, key=lambda x: x[0], reverse=True)
        history.append({"generation": gen+1, "best_score": scored[0][0], **scored[0][1]})

        elites = [ind for _, ind in scored[:elite_size]]
        new_population = elites.copy()

        while len(new_population) < population_size:
            p1, p2 = random.sample(elites, 2)
            child = crossover(p1, p2)
            child = mutate(child)
            new_population.append(child)

        population = new_population

    best_score, best_individual = sorted(scored, key=lambda x: x[0], reverse=True)[0]
    return best_individual, pd.DataFrame(history)

## 9. Lancement de l'optimisation génétique

Pour un test rapide, gardez une population et un nombre de générations faibles. Pour un vrai entraînement, augmentez progressivement ces valeurs.

In [ ]:
# best_params, ga_history = genetic_search(population_size=6, generations=3, elite_size=2)
# print("Best parameters:", best_params)
# ga_history

## 10. Entraînement final avec les meilleurs hyperparamètres

Si vous n'avez pas lancé l'algorithme génétique, vous pouvez utiliser les paramètres par défaut ci-dessous.

In [ ]:
best_params = {
    "learning_rate": 3e-4,
    "dropout": 0.3,
    "batch_size": 16,
    "freeze_backbone": True
}

EPOCHS_FINAL = 10

# train_dataset, val_dataset, test_dataset, train_loader, val_loader, test_loader = load_datasets(best_params["batch_size"])
# model = build_model(dropout=best_params["dropout"], freeze_backbone=best_params["freeze_backbone"])
# criterion = nn.CrossEntropyLoss()
# optimizer = optim.Adam(filter(lambda p: p.requires_grad, model.parameters()), lr=best_params["learning_rate"])

# history = {"train_loss": [], "train_acc": [], "val_loss": [], "val_acc": []}

# for epoch in range(EPOCHS_FINAL):
#     train_loss, train_acc = train_one_epoch(model, train_loader, criterion, optimizer)
#     val_loss, val_acc, _, _ = evaluate(model, val_loader, criterion)
#     
#     history["train_loss"].append(train_loss)
#     history["train_acc"].append(train_acc)
#     history["val_loss"].append(val_loss)
#     history["val_acc"].append(val_acc)
#     
#     print(f"Epoch {epoch+1}/{EPOCHS_FINAL} | Train Acc: {train_acc:.3f} | Val Acc: {val_acc:.3f}")

# torch.save(model.state_dict(), RESULTS_DIR / "atlasvision_ai_model.pth")

## 11. Courbes d'entraînement

In [ ]:
# plt.figure(figsize=(8, 5))
# plt.plot(history["train_acc"], label="Train accuracy")
# plt.plot(history["val_acc"], label="Validation accuracy")
# plt.xlabel("Epoch")
# plt.ylabel("Accuracy")
# plt.title("AtlasVision AI - Training Curves")
# plt.legend()
# plt.grid(True)
# plt.savefig(RESULTS_DIR / "training_curves.png", dpi=300, bbox_inches="tight")
# plt.show()

## 12. Évaluation finale sur test set

In [ ]:
# test_loss, test_acc, y_true, y_pred = evaluate(model, test_loader, criterion)
# print("Test accuracy:", test_acc)
# print(classification_report(y_true, y_pred, target_names=CLASS_NAMES))

# cm = confusion_matrix(y_true, y_pred)
# disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=CLASS_NAMES)
# disp.plot(cmap="Blues", values_format="d")
# plt.title("AtlasVision AI - Confusion Matrix")
# plt.savefig(RESULTS_DIR / "confusion_matrix.png", dpi=300, bbox_inches="tight")
# plt.show()

## 13. Grad-CAM pour explicabilité

Grad-CAM permet de visualiser les régions de l'image qui contribuent le plus à la décision du CNN. Cette étape est essentielle dans un projet d'IA médicale, car elle améliore l'interprétabilité et la crédibilité clinique du modèle.

In [ ]:
class GradCAM:
    def __init__(self, model, target_layer):
        self.model = model
        self.target_layer = target_layer
        self.gradients = None
        self.activations = None
        self._register_hooks()

    def _register_hooks(self):
        def forward_hook(module, input, output):
            self.activations = output.detach()

        def backward_hook(module, grad_input, grad_output):
            self.gradients = grad_output[0].detach()

        self.target_layer.register_forward_hook(forward_hook)
        self.target_layer.register_full_backward_hook(backward_hook)

    def generate(self, image_tensor, class_idx=None):
        self.model.eval()
        output = self.model(image_tensor.unsqueeze(0).to(device))
        if class_idx is None:
            class_idx = output.argmax(dim=1).item()

        self.model.zero_grad()
        output[0, class_idx].backward()

        gradients = self.gradients[0]
        activations = self.activations[0]
        weights = gradients.mean(dim=(1, 2))
        cam = torch.zeros(activations.shape[1:], dtype=torch.float32).to(device)

        for i, w in enumerate(weights):
            cam += w * activations[i]

        cam = torch.relu(cam)
        cam = cam - cam.min()
        cam = cam / (cam.max() + 1e-8)
        return cam.cpu().numpy(), class_idx

## 14. Visualisation Grad-CAM sur une image de test

In [ ]:
# import cv2

# image_tensor, label = test_dataset[0]
# target_layer = model.layer4[-1]
# gradcam = GradCAM(model, target_layer)
# cam, predicted_class = gradcam.generate(image_tensor)

# img = image_tensor.permute(1, 2, 0).numpy()
# img = img * np.array([0.229, 0.224, 0.225]) + np.array([0.485, 0.456, 0.406])
# img = np.clip(img, 0, 1)

# cam_resized = cv2.resize(cam, (IMG_SIZE, IMG_SIZE))

# plt.figure(figsize=(10, 4))
# plt.subplot(1, 2, 1)
# plt.imshow(img)
# plt.title(f"Original - True: {CLASS_NAMES[label]}")
# plt.axis("off")

# plt.subplot(1, 2, 2)
# plt.imshow(img)
# plt.imshow(cam_resized, alpha=0.45, cmap="jet")
# plt.title(f"Grad-CAM - Pred: {CLASS_NAMES[predicted_class]}")
# plt.axis("off")

# plt.savefig(RESULTS_DIR / "gradcam_example.png", dpi=300, bbox_inches="tight")
# plt.show()

## 15. Conclusion pour README GitHub

**AtlasVision AI** démontre une chaîne complète d'aide au diagnostic en mammographie : prétraitement, transfert d'apprentissage, optimisation évolutive par algorithme génétique, évaluation quantitative et explicabilité visuelle par Grad-CAM.

### Message à mettre dans le dépôt

> AtlasVision AI combines convolutional neural networks and genetic algorithms to support explainable mammography classification for breast cancer screening.